# QFF 2026 Phase 2 — PS-S1 Quantum Simulation

This notebook reproduces the shortlisted team's Phase 2 PS-S1 deliverables and common Processor D Bell study. The official `qff_sim.py`, locked instance files, processor definitions, seeds, and noise settings are used as supplied.

Set `RECOMPUTE = True` to run the VQE layer scan, noisy D cases, resource compilation, and Bell circuits. The saved CSVs and figures are loaded by default. Use `RESUME = True` to continue saved checkpoints, `N6_ONLY = True` to preserve N=4 outputs and run only the optional bonus tier, or `SUPPLEMENT_ONLY = True` to regenerate the seed and noisy A/B/D comparison tables from saved optimizer results.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, HTML, Markdown

ROOT = Path.cwd()
while not (ROOT / 'qff_sim.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'qff_sim.py').exists():
    raise FileNotFoundError('Open this notebook from the submission repository.')
sys.path.insert(0, str(ROOT))
from scripts.run_phase2_s1 import run_phase2_s1, generate_phase2_outputs, TABLES_DIR, FIGURES_DIR

RECOMPUTE = False
INCLUDE_N6 = False
RESUME = False
N6_ONLY = False
SUPPLEMENT_ONLY = False
MAX_LAYERS = 12
if RECOMPUTE:
    results = generate_phase2_outputs(max_layers=MAX_LAYERS, include_n6=INCLUDE_N6, resume=RESUME, n6_only=N6_ONLY, supplement_only=SUPPLEMENT_ONLY)
else:
    results = run_phase2_s1()
print('Phase 2 PS-S1 results loaded/generated.')
print('Minimum N=4 layers by g:', results.get('n4_min_layers', {}))


In [ ]:
import csv, html

def show_csv(path, limit=100):
    with open(path, newline='', encoding='utf-8-sig') as f:
        rows = list(csv.reader(f))
    headers = rows[0]
    shown = rows[1:limit+1]
    head = ''.join(f'<th>{html.escape(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(v)}</td>' for v in row) + '</tr>' for row in shown)
    display(HTML(f'<div style="overflow-x:auto"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'))

print('Bell-state results')
show_csv(TABLES_DIR / 'bell.csv')
print('PS-S1 HVA layer/result table')
show_csv(TABLES_DIR / 's1_results.csv')
print('Noisy Processor D results')
show_csv(TABLES_DIR / 's1_noisy.csv')
for filename, title in [('s1_seed_summary.csv', 'Three official seeds at the selected layer'), ('s1_noisy_architectures.csv', 'Supplementary noisy A/B/D comparison')]:
    path = TABLES_DIR / filename
    print(title)
    if path.exists():
        show_csv(path)
    else:
        print('Not generated yet. Run: python scripts/run_phase2_s1.py --supplement-only')


## Figures

The plots show observables across the finite-size crossover, minimum HVA depth, measured A/B/D resources, Bell fidelity with one standard error, the frozen D topology, and (when generated) the supplementary noisy A/B/D comparison.

In [ ]:
for name in ['phase2_s1_energy.png', 'phase2_s1_observables.png', 'phase2_s1_layer_study.png', 'phase2_s1_resources.png', 'phase2_bell_fidelity.png', 'processor_D_topology.png', 'phase2_s1_noisy_architectures.png']:
    path = FIGURES_DIR / name
    if path.exists():
        display(Markdown(f'### {name.replace("_", " ")}'))
        display(Image(filename=str(path)))


## Reproduction notes

- Required package versions are pinned in `requirements.txt`.
- `python scripts/run_phase2_s1.py` regenerates the required N=4 tables and figures.
- `python scripts/run_phase2_s1.py --include-n6 --resume` runs or resumes the optional N=6 tier.
- `python scripts/run_phase2_s1.py --n6-only --resume` keeps the saved N=4 work and spends compute on N=6 only.
- `python scripts/run_phase2_s1.py --supplement-only` regenerates per-seed summaries and the supplemental noisy A/B/D comparison without rerunning VQE.
- `python tools/check_processor_d.py processors/processor_D.json 6` validates the frozen D design.
- All noisy device results are Aer simulations using the official synthetic model; no QPU claim is made.
